Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import numpy as np
import pandas as pd
from pathlib import Path
import sys

sys.path.append(os.path.join(Path.cwd(), "VTFW_Lab4"))

from VTFW_Lab4.data.data_writer import get_data_dir
from VTFW_Lab4.data.data_writer import get_data_dir, write_flying_wing
from VTFW_Lab4.geometry.naca_airfoil import NACA4, NACA5
from VTFW_Lab4.geometry.generators import get_wing_geometries, create_trapezoidal_dimensionalized_wing
from VTFW_Lab4.data.data_classes import FlyingWing
from VTFW_Lab4.data.data_reader import read_flying_wing, read_list_of_flying_wings
from VTFW_Lab4.utils.common import find_simulation
from VTFW_Lab4.utils.common import find_analysis
from VTFW_Lab4.geometry.wing_geometry import get_trapezoidal_aspect_ratio, get_taper_ratio, get_sweep_angle
from VTFW_Lab4.utils.aerosandbox_interface import run_asb_vlm
from VTFW_Lab4.analysis.cg_boundary_analysis import get_cg_for_trim
from VTFW_Lab4.analysis.cg_boundary_analysis import get_cm_cg, get_cg_for_trim, get_lin_cg_for_trim, get_lin_cm_cg
# Initialise data directory for data storage 

data_dir_name = f"Bachelor_Thesis_Data_Dir_Validation"

vlm_simulation_id = "bat_vlm_simu_1"
vlm_analysis_id = "bat_vlm_analysis_2"


data_dir = get_data_dir(data_dir_name, os.path.join(Path.cwd(), "data"))

#os.startfile(data_dir)


Analysis

In [ ]:


flying_wing = FlyingWing("good_fw",  create_trapezoidal_dimensionalized_wing(1.0, 10, 30, NACA4("1415")), [], [])


flying_wing.wing_geometry.current_total_twist = 0

x_cg = 1.662 

def enforce_twist_boundaries(twist):
    if twist < -5:
        return -5
    elif twist > 5:
        return 5
    return twist


alpha_values = np.linspace(0, 10, 11)

_0_twist_cm_values = []
p_controll_cm_values_4 = []
p_controll_cm_values_8 = []

twist_vals_4 = []
twist_vals_8 = []
for alpha in alpha_values:

    flying_wing.wing_geometry.current_total_twist = 0
    
    simu = run_asb_vlm(flying_wing, "moment_curve_simulation", 100, alpha, 0)
    _0_twist_cm_values.append(get_cm_cg(simu, x_cg))

    
    flying_wing.wing_geometry.current_total_twist = enforce_twist_boundaries(-0.7*(4 - alpha) - 1.43)
    twist_vals_4.append(flying_wing.wing_geometry.current_total_twist)
    simu = run_asb_vlm(flying_wing, "moment_curve_simulation", 100, alpha, 0)
    p_controll_cm_values_4.append(get_cm_cg(simu, x_cg))

    flying_wing.wing_geometry.current_total_twist = enforce_twist_boundaries(-0.5*(8 - alpha) - 0.75)
    twist_vals_8.append(flying_wing.wing_geometry.current_total_twist)
    simu = run_asb_vlm(flying_wing, "moment_curve_simulation", 100, alpha, 0)
    p_controll_cm_values_8.append(get_cm_cg(simu, x_cg))


print(f"Max twist for alpha0 at 4 {max(twist_vals_4)}")
print(f"Min twist for alpha0 at 4 {min(twist_vals_4)}")

print(f"Max twist for alpha0 at 8 {max(twist_vals_8)}")
print(f"Min twist for alpha0 at 8 {min(twist_vals_8)}")

plt.plot(alpha_values, _0_twist_cm_values, "-o", color = "red", label="Open loop untwisted wing")
plt.plot(alpha_values, p_controll_cm_values_4, "-o", color = "green", label="Controller for target alpha = 4°")
plt.plot(alpha_values, p_controll_cm_values_8, "-o", color = "blue", label="Controller for target alpha = 8°")
plt.legend()
plt.xlabel("Angel of atack")
plt.ylabel("Moment coefficent about CG")
plt.grid(True)

In [ ]:


flying_wing = FlyingWing("bad_fw", create_trapezoidal_dimensionalized_wing(0.2, 4, 0, NACA4("4515")), [], [])


flying_wing.wing_geometry.current_total_twist = 0

x_cg = get_cg_for_trim(run_asb_vlm(flying_wing, "trim_cond_simulation", 100, 4, 0))

print(f"Calculated center of gravity postion for angle of atack at 4: {x_cg}")

alpha_values = np.linspace(0, 10, 11)
twist_values = np.linspace(-5, 5, 11)

_0_twist_cm_values = []
p_controll_cm_values_4 = []
p_controll_cm_values_8 = []

for alpha in alpha_values:
    flying_wing.wing_geometry.current_total_twist = 0
    
    simu = run_asb_vlm(flying_wing, "moment_curve_simulation", 100, alpha, 0)
    _0_twist_cm_values.append(get_cm_cg(simu, x_cg))

    flying_wing.wing_geometry.current_total_twist = enforce_twist_boundaries(3*(4 - alpha))
    
    simu = run_asb_vlm(flying_wing, "moment_curve_simulation", 100, alpha, 0)
    p_controll_cm_values_4.append(get_cm_cg(simu, x_cg))


    flying_wing.wing_geometry.current_total_twist = enforce_twist_boundaries(4*(8 - alpha))
    
    simu = run_asb_vlm(flying_wing, "moment_curve_simulation", 100, alpha, 0)
    p_controll_cm_values_8.append(get_cm_cg(simu, x_cg))


cm_cgs = []
for twist in twist_values:
    flying_wing.wing_geometry.current_total_twist = twist
    simu = run_asb_vlm(flying_wing, "moment_curve_simulation", 100, 8, 0)
    cm_cgs.append(get_cm_cg(simu, x_cg))

print(f"Minimal achivable moment coefficent about center of grabity: {min(cm_cgs)}")

plt.plot(alpha_values, _0_twist_cm_values, "-o", color = "red", label="Open loop untwisted wing")
plt.plot(alpha_values, p_controll_cm_values_4, "-o", color = "green", label="Controller for target alpha = 4°")
plt.plot(alpha_values, p_controll_cm_values_8, "-o", color = "blue", label="Controller for target alpha = 8°")
plt.legend()
plt.xlabel("Angel of atack")
plt.ylabel("Moment coefficent about CG")
plt.grid(True)